# Estimate photons_per_unit from the data

This notebook estimates the effective count scale used by <code>utils/salt.py</code> and shows why the estimate is (or is not) trustworthy.

For a signal in image units,

$$
Y = \frac{\operatorname{Poisson}(p\mu)}{p},
\qquad
\operatorname{Var}(Y\mid\mu) = c + \frac{\mu}{p},
$$

so the slope of variance versus mean is $s=1/p$. The fitted quantity is an **effective detected-count scale**; with a PMT, HyD, EMCCD, or other multiplication detector it need not equal incident photons.

The notebook deliberately shows two fits:

1. **Legacy diagnostic:** raw pair variance versus the noisy raw pair mean. This is the fit that previously gave about 0.000735 in raw units and a strongly curved plot.
2. **Recommended estimate:** the same raw pair variance, but binned by the corresponding denoised SUPPORT intensity. SUPPORT is used only as the intensity coordinate. The variance is still measured from raw frames.

Run the notebook from top to bottom. Outputs are intentionally cleared because the frame-selection logic changed; any result must come from a fresh run. The saved paths below are your current dataset, while the synthetic switch provides a quick self-test on another machine.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile

# Make figures appear both interactively and in nbconvert-executed copies.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except (AttributeError, NameError):
    pass

# ----------------------------
# Input configuration
# ----------------------------
USE_SYNTHETIC_DEMO = False
RAW_PATH = Path('/Volumes/NVRserverStoragePool2/Josh_Garfinkel/01_projects/01_dinamiCa/20260806_validationSet/01_data/01_raw/6xFAD_TRD_GC8_M205_20251119_FOV1_4D/6xFAD_TRD_GC8_M205_20251119_FOV1_4D_Ch2.tif')
SUPPORT_REFERENCE_PATH = Path('/Volumes/NVRserverStoragePool2/Josh_Garfinkel/01_projects/01_dinamiCa/20260806_validationSet/01_data/02_processed/01_denoised/6xFAD_TRD_GC8_M205_20251119_FOV1_4D/SPRT20261008/6xFAD_TRD_GC8_M205_20251119_FOV1_4D_Ch2_SPRT.tif')

# Axis, central frame window, and ROI controls. TIME_AXIS=0 matches salt.py.
TIME_AXIS = 0
FRAME_START = 60
FRAME_END_MARGIN = 60
MAX_PAIRS = 50
Z_SLICE = slice(None)
Y_SLICE = slice(None)
X_SLICE = slice(None)
Z_STRIDE = 1
XY_STRIDE = 1

# Detector limits and fit controls.
SATURATION_VALUE = None
FLOOR_VALUE = None
MIN_MEAN_LEVEL = 0.0
MAX_MEAN_LEVEL = None
FIT_QUANTILES = (0.02, 0.98)
PRIMARY_FIT_QUANTILES = (0.01, 0.99)
SCALE_FIT_QUANTILES = (0.01, 0.99)
N_BINS = 30
MIN_SAMPLES_PER_BIN = 500
MAX_SAMPLES = 2_000_000
RANDOM_SEED = 7

# Laptop-friendly diagnostics.
DIAGNOSTIC_SAMPLE_COUNT = 40_000
TIME_BLOCKS = 4
DIAGNOSTIC_LAGS = (1, 2, 4, 8, 16)
LAG_COORDINATE_COUNT = 10_000
TILE_GRID = (4, 4)
SPATIAL_PAIR_COUNT = 6
SPATIAL_CORRELATION_LAGS = 8
QUIET_SENSITIVITY_QUANTILE = 0.50
MAXIMUM_QUIET_RELATIVE_DIFFERENCE = 0.15
DARK_BORDER_FRACTION = 0.10
DARK_BORDER_QUANTILE = 0.20
MODEL_R2_WARNING_THRESHOLD = 0.98
QUADRATIC_AICC_WARNING = 6.0

# Values included only for comparison with earlier runs.
CURRENT_SALT_PHOTONS_PER_UNIT = 0.01
PREVIOUS_RAW_PAIR_ESTIMATE = 0.00073539555
PREVIOUS_SCALE_ADJUSTED_ESTIMATE = 0.00046749053

# Synthetic self-test. The central-window margins are ignored in this mode.
SYNTHETIC_PHOTONS_PER_UNIT = 0.02
SYNTHETIC_READ_NOISE_SD = 15.0
SYNTHETIC_FRAMES = 40

## What is being fitted, and why

For adjacent raw frames $A$ and $B$ with the same latent signal,

$$
D=A-B,\qquad v=\frac{\operatorname{Var}(D)}{2}.
$$

Fixed spatial structure cancels in $D$. Within every frame-pair/intensity-bin group, the notebook also subtracts the mean of $D$ so synchronized offsets do not inflate the variance.

The old analysis used $M=(A+B)/2$ on the horizontal axis. For an ideal Poisson process this construction has a useful conditional identity, so it is not automatically wrong. In this dataset, however, $M$ is extremely noisy and mixes detector noise, biological change, and the quantity being squared on the vertical axis. The visible upward curvature and poor residuals show that a single straight-line interpretation of that plot is unsafe.

The recommended horizontal coordinate is instead obtained from the matching SUPPORT output. First, temporal means at fixed voxels estimate the affine scale

$$
S = a + gR,
$$

where $R$ is raw intensity and $S$ is SUPPORT intensity. Both the scale fit and pair selection use the central, end-exclusive frame window $[\texttt{FRAME_START}, T-\texttt{FRAME_END_MARGIN})$. The scale fit uses temporal means from every frame in that window and excludes the outer 1% of intensity values at each end, where the empirical mapping is mildly nonlinear. Up to `MAX_PAIRS` non-overlapping adjacent pairs are distributed evenly across the same window. A SUPPORT pair level is then mapped back to raw-equivalent units as $(S-a)/g$. The raw variance is fit as

$$
v = c+sR,\qquad p_{\rm raw}=1/s,\qquad p_{\rm SUPPORT}=p_{\rm raw}/g.
$$

The zero-variance crossing $-c/s$ is reported as an **effective pedestal**, not a measured detector black level. Read noise, offsets, and model mismatch all contribute to it; dark/laser-off frames are needed to identify a physical black level.

Both a line and $v=c+sR+qR^2$ are fit to the same binned points. The notebook reports $\Delta\mathrm{AICc}=\mathrm{AICc}_{linear}-\mathrm{AICc}_{quadratic}$: values above about 6 favor the quadratic model, while negative values favor the line. The sign matters when comparing with Stefanov's signal-dependent charge-sharing model: charge sharing predicts a *negative* quadratic term (downward/sub-Poisson curvature), not the positive curvature seen in the old noisy-axis plot. The AICc comparison is an unweighted heuristic for these heteroskedastic binned variances. Because adjacent frames can also differ through motion or biology, the quietest half of locations by SUPPORT-predicted change is fit as a sensitivity check. A difference above 15% makes the result provisional but does not silently replace the primary estimate.

In [ ]:
def read_tiff(path):
    path = Path(path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(path)
    with tifffile.TiffFile(path) as tif:
        axes = tif.series[0].axes
    try:
        array = tifffile.memmap(path, series=0, mode='r')
        method = 'memory-mapped'
    except (OSError, ValueError):
        array = tifffile.imread(path)
        method = 'loaded into memory'
    print(f'{path.name}: {method}, axes={axes!r}, shape={array.shape}, dtype={array.dtype}')
    return array, axes


def move_time_first(array, axes, configured_axis):
    if configured_axis is None:
        if axes is None or 'T' not in axes:
            raise ValueError('TIFF metadata has no T axis; set TIME_AXIS explicitly.')
        time_axis = axes.index('T')
    else:
        time_axis = int(configured_axis)
    if time_axis < -array.ndim or time_axis >= array.ndim:
        raise ValueError(f'TIME_AXIS={time_axis} is invalid for shape {array.shape}.')
    if axes and 'T' not in axes:
        print(
            f'WARNING: TIFF axes are {axes!r}; treating axis {time_axis} as time. '
            'Confirm that it is not Z.'
        )
    return np.moveaxis(array, time_axis, 0)


def crop_time_first(array):
    if array.ndim == 3:
        return array[:, Y_SLICE, X_SLICE][:, ::XY_STRIDE, ::XY_STRIDE]
    if array.ndim == 4:
        cropped = array[:, Z_SLICE, Y_SLICE, X_SLICE]
        if cropped.ndim == 3:
            return cropped[:, ::XY_STRIDE, ::XY_STRIDE]
        return cropped[:, ::Z_STRIDE, ::XY_STRIDE, ::XY_STRIDE]
    raise ValueError(
        f'Expected time-first TYX or TZYX data, but received shape {array.shape}.'
    )


def make_synthetic_stack():
    rng = np.random.default_rng(RANDOM_SEED)
    height = width = 128
    yy, xx = np.mgrid[-1:1:complex(height), -1:1:complex(width)]
    clean_frame = (
        120.0
        + 1_400.0 * (xx + 1.0) / 2.0
        + 1_500.0 * np.exp(-((xx + 0.35) ** 2 + (yy - 0.20) ** 2) / 0.08)
        + 900.0 * np.exp(-((xx - 0.40) ** 2 + (yy + 0.35) ** 2) / 0.04)
    ).astype(np.float32)
    clean = np.broadcast_to(
        clean_frame, (SYNTHETIC_FRAMES, height, width)
    ).copy()
    raw = rng.poisson(clean * SYNTHETIC_PHOTONS_PER_UNIT).astype(np.float32)
    raw /= SYNTHETIC_PHOTONS_PER_UNIT
    raw += rng.normal(
        0.0, SYNTHETIC_READ_NOISE_SD, raw.shape
    ).astype(np.float32)
    return raw, clean

In [ ]:
if USE_SYNTHETIC_DEMO:
    raw, synthetic_clean = make_synthetic_stack()
    support_reference = synthetic_clean
    source_description = 'synthetic demonstration'
    known_photons_per_unit = SYNTHETIC_PHOTONS_PER_UNIT
    analysis_frame_start = 0
    analysis_frame_stop = raw.shape[0]
else:
    raw_on_disk, raw_axes = read_tiff(RAW_PATH)
    raw = crop_time_first(move_time_first(raw_on_disk, raw_axes, TIME_AXIS))
    if SUPPORT_REFERENCE_PATH is None:
        support_reference = None
    else:
        reference_on_disk, reference_axes = read_tiff(SUPPORT_REFERENCE_PATH)
        support_reference = crop_time_first(
            move_time_first(reference_on_disk, reference_axes, TIME_AXIS)
        )
        if support_reference.shape != raw.shape:
            raise ValueError(
                'Raw and SUPPORT TIFF shapes differ after axis/ROI selection: '
                f'{raw.shape} versus {support_reference.shape}.'
            )
    source_description = str(Path(RAW_PATH).expanduser())
    known_photons_per_unit = None
    analysis_frame_start = FRAME_START
    analysis_frame_stop = raw.shape[0] - FRAME_END_MARGIN

if not 0 <= analysis_frame_start < analysis_frame_stop <= raw.shape[0]:
    raise ValueError(
        'The central frame window is empty or outside the movie: '
        f'[{analysis_frame_start}, {analysis_frame_stop}) for T={raw.shape[0]}.'
    )
available_pairs = (analysis_frame_stop - analysis_frame_start) // 2
pair_count = available_pairs if MAX_PAIRS is None else min(available_pairs, MAX_PAIRS)
if pair_count < 2:
    raise ValueError('The selected range contains fewer than two complete frame pairs.')
candidate_pair_ids = np.arange(available_pairs, dtype=np.int64)
if pair_count == available_pairs:
    selected_pair_ids = candidate_pair_ids
else:
    selected_pair_ids = candidate_pair_ids[
        np.linspace(0, available_pairs - 1, pair_count, dtype=np.int64)
    ]
pair_starts = analysis_frame_start + 2 * selected_pair_ids
pair_frame_indices = np.column_stack(
    (pair_starts, pair_starts + 1)
).reshape(-1)
scale_frame_indices = np.arange(
    analysis_frame_start, analysis_frame_stop, dtype=np.int64
)
# Traces below contain only scale_frame_indices, so convert global TIFF indices
# to positions local to that sampled central-window trace.
pair_trace_indices = pair_frame_indices - analysis_frame_start

print(f'Analyzing {source_description}')
if USE_SYNTHETIC_DEMO:
    print('SYNTHETIC CHECK ONLY: do not copy this estimate into a real experiment.')
print(f'Time-first shape after ROI selection: {raw.shape}')
print(
    f'Central calibration window: [{analysis_frame_start}, '
    f'{analysis_frame_stop}) ({scale_frame_indices.size} frames)'
)
print(
    f'Using {pair_count} of {available_pairs} non-overlapping adjacent pairs, '
    f'distributed from frames {pair_starts[0]}-{pair_starts[0] + 1} to '
    f'{pair_starts[-1]}-{pair_starts[-1] + 1}'
)
print('Using every central-window frame for the temporal-mean scale fit')
print(f'SUPPORT reference supplied: {support_reference is not None}')

preview_index = int(pair_frame_indices[0])
preview_images = [('Raw', np.asarray(raw[preview_index], dtype=np.float32))]
if support_reference is not None:
    preview_images.append(
        ('SUPPORT', np.asarray(support_reference[preview_index], dtype=np.float32))
    )

fig, axes = plt.subplots(1, len(preview_images), figsize=(6 * len(preview_images), 5))
axes = np.atleast_1d(axes)
for axis, (label, volume) in zip(axes, preview_images):
    image = volume[volume.shape[0] // 2] if volume.ndim == 3 else volume
    finite = image[np.isfinite(image)]
    if finite.size == 0:
        raise ValueError(f'{label} preview has no finite values.')
    vmin, vmax = np.percentile(finite, [1, 99])
    shown = axis.imshow(image, cmap='gray', vmin=vmin, vmax=vmax)
    axis.set_title(f'{label}, frame {preview_index}')
    axis.set_axis_off()
    fig.colorbar(shown, ax=axis, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

In [ ]:
def correlation(x, y):
    x = np.asarray(x, dtype=np.float64).ravel()
    y = np.asarray(y, dtype=np.float64).ravel()
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3:
        return np.nan
    x = x[valid]
    y = y[valid]
    x -= x.mean()
    y -= y.mean()
    denominator = np.sqrt(np.dot(x, x) * np.dot(y, y))
    return float(np.dot(x, y) / denominator) if denominator > 0 else np.nan


def aicc(residual_sum_squares, sample_count, parameter_count):
    residual_sum_squares = max(
        float(residual_sum_squares), np.finfo(np.float64).tiny
    )
    base = sample_count * np.log(residual_sum_squares / sample_count)
    base += 2.0 * parameter_count
    correction_denominator = sample_count - parameter_count - 1
    if correction_denominator <= 0:
        return np.inf
    return float(
        base
        + 2.0 * parameter_count * (parameter_count + 1)
        / correction_denominator
    )


def fit_line_and_quadratic(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    if x.size < 5:
        raise ValueError('At least five points are needed for model comparison.')
    slope, intercept = np.polyfit(x, y, 1)
    linear_prediction = intercept + slope * x

    center = float(np.mean(x))
    scale = float(np.std(x))
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError('The predictor has no usable intensity range.')
    normalized_x = (x - center) / scale
    normalized_q2, normalized_q1, normalized_q0 = np.polyfit(
        normalized_x, y, 2
    )
    quadratic_prediction = (
        normalized_q0
        + normalized_q1 * normalized_x
        + normalized_q2 * normalized_x ** 2
    )
    q2 = normalized_q2 / scale ** 2
    q1 = normalized_q1 / scale - 2.0 * normalized_q2 * center / scale ** 2
    q0 = (
        normalized_q0
        - normalized_q1 * center / scale
        + normalized_q2 * center ** 2 / scale ** 2
    )

    total_sum = float(np.sum((y - y.mean()) ** 2))
    linear_rss = float(np.sum((y - linear_prediction) ** 2))
    quadratic_rss = float(np.sum((y - quadratic_prediction) ** 2))
    linear_r2 = 1.0 - linear_rss / total_sum if total_sum > 0 else np.nan
    quadratic_r2 = 1.0 - quadratic_rss / total_sum if total_sum > 0 else np.nan
    linear_aicc = aicc(linear_rss, x.size, 3)
    quadratic_aicc = aicc(quadratic_rss, x.size, 4)
    return {
        'slope': float(slope),
        'intercept': float(intercept),
        'linear_prediction': linear_prediction,
        'linear_r2': float(linear_r2),
        'linear_aicc': linear_aicc,
        'q2': float(q2),
        'q1': float(q1),
        'q0': float(q0),
        'quadratic_prediction': quadratic_prediction,
        'quadratic_r2': float(quadratic_r2),
        'quadratic_aicc': quadratic_aicc,
        'delta_aicc': float(linear_aicc - quadratic_aicc),
    }


def sample_matched_traces(raw_stack, reference_stack, frame_indices):
    spatial_shape = tuple(raw_stack.shape[1:])
    spatial_count = int(np.prod(spatial_shape))
    sample_count = min(int(DIAGNOSTIC_SAMPLE_COUNT), spatial_count)
    rng = np.random.default_rng(RANDOM_SEED)
    spatial_indices = np.sort(
        rng.choice(spatial_count, size=sample_count, replace=False)
    )
    raw_trace = np.empty(
        (frame_indices.size, sample_count), dtype=np.float32
    )
    reference_trace = (
        np.empty_like(raw_trace) if reference_stack is not None else None
    )
    progress_step = max(1, frame_indices.size // 5)
    for local_index, frame_index in enumerate(frame_indices):
        raw_frame = np.asarray(raw_stack[int(frame_index)], dtype=np.float32)
        raw_trace[local_index] = np.take(raw_frame, spatial_indices)
        if reference_trace is not None:
            reference_frame = np.asarray(
                reference_stack[int(frame_index)], dtype=np.float32
            )
            reference_trace[local_index] = np.take(
                reference_frame, spatial_indices
            )
        if (
            local_index == 0
            or local_index + 1 == frame_indices.size
            or (local_index + 1) % progress_step == 0
        ):
            print(
                f'  sampled {local_index + 1:>3}/{frame_indices.size} frames',
                end='\r' if local_index + 1 < frame_indices.size else '\n',
            )
    coordinates = np.unravel_index(spatial_indices, spatial_shape)
    return raw_trace, reference_trace, coordinates, spatial_indices


def temporal_mean(values):
    values = np.asarray(values, dtype=np.float64)
    finite = np.isfinite(values)
    counts = finite.sum(axis=0)
    totals = np.where(finite, values, 0.0).sum(axis=0)
    result = np.full(values.shape[1], np.nan, dtype=np.float64)
    valid = counts > 0
    result[valid] = totals[valid] / counts[valid]
    return result


def fit_intensity_scale(
    raw_values,
    reference_values,
    coordinate_mask=None,
    n_bins=N_BINS,
    min_samples=50,
    fit_quantiles=SCALE_FIT_QUANTILES,
):
    if reference_values is None:
        return None
    raw_means = temporal_mean(raw_values)
    reference_means = temporal_mean(reference_values)
    valid = np.isfinite(raw_means) & np.isfinite(reference_means)
    if coordinate_mask is not None:
        valid &= np.asarray(coordinate_mask, dtype=bool)
    if fit_quantiles is not None:
        lower, upper = np.quantile(reference_means[valid], fit_quantiles)
        valid &= (reference_means >= lower) & (reference_means <= upper)
    raw_means = raw_means[valid]
    reference_means = reference_means[valid]
    if raw_means.size < max(100, 5 * min_samples):
        raise ValueError('Too few spatial samples remain for the intensity-scale fit.')

    edges = np.unique(
        np.quantile(reference_means, np.linspace(0.0, 1.0, n_bins + 1))
    )
    indices = np.clip(
        np.searchsorted(edges, reference_means, side='right') - 1,
        0,
        edges.size - 2,
    )
    binned_raw, binned_reference, counts = [], [], []
    for bin_index in range(edges.size - 1):
        selected = indices == bin_index
        count = int(selected.sum())
        if count < min_samples:
            continue
        binned_raw.append(float(np.mean(raw_means[selected])))
        binned_reference.append(float(np.mean(reference_means[selected])))
        counts.append(count)
    binned_raw = np.asarray(binned_raw, dtype=np.float64)
    binned_reference = np.asarray(binned_reference, dtype=np.float64)
    if binned_raw.size < 5:
        raise ValueError('Too few populated bins remain for the intensity-scale fit.')
    models = fit_line_and_quadratic(binned_raw, binned_reference)
    if not np.isfinite(models['slope']) or models['slope'] <= 0:
        raise ValueError('The raw-to-SUPPORT scale gain is not positive.')
    return {
        'gain': models['slope'],
        'offset': models['intercept'],
        'r_squared': models['linear_r2'],
        'models': models,
        'raw_means': raw_means,
        'reference_means': reference_means,
        'binned_raw': binned_raw,
        'binned_reference': binned_reference,
        'counts': np.asarray(counts),
    }


def make_nonoverlap_pair_vectors(
    raw_values,
    reference_values,
    scale_offset,
    scale_gain,
):
    usable_frames = 2 * (raw_values.shape[0] // 2)
    raw_values = raw_values[:usable_frames]
    raw_a = raw_values[0::2]
    raw_b = raw_values[1::2]
    raw_difference = raw_a - raw_b
    raw_pair_mean = 0.5 * (raw_a + raw_b)
    total = np.sum(raw_values, axis=0, dtype=np.float64)
    leave_two_out = (
        total[None, :] - raw_a.astype(np.float64) - raw_b.astype(np.float64)
    ) / max(usable_frames - 2, 1)

    if reference_values is not None:
        reference_values = reference_values[:usable_frames]
        reference_a = reference_values[0::2]
        reference_b = reference_values[1::2]
        reference_pair_mean = 0.5 * (reference_a + reference_b)
        reference_difference = reference_a - reference_b
        support_raw_equivalent = (
            reference_pair_mean - scale_offset
        ) / scale_gain
    else:
        reference_difference = np.full_like(raw_difference, np.nan)
        support_raw_equivalent = leave_two_out

    valid = (
        np.isfinite(raw_a)
        & np.isfinite(raw_b)
        & np.isfinite(raw_difference)
        & np.isfinite(support_raw_equivalent)
    )
    if SATURATION_VALUE is not None:
        valid &= (raw_a < SATURATION_VALUE) & (raw_b < SATURATION_VALUE)
    if FLOOR_VALUE is not None:
        valid &= (raw_a > FLOOR_VALUE) & (raw_b > FLOOR_VALUE)

    pair_ids = np.broadcast_to(
        np.arange(raw_difference.shape[0], dtype=np.int32)[:, None],
        raw_difference.shape,
    )
    return {
        'raw_pair_mean': raw_pair_mean[valid].astype(np.float32, copy=False),
        'support_raw_equivalent': support_raw_equivalent[valid].astype(
            np.float32, copy=False
        ),
        'leave_two_out': leave_two_out[valid].astype(np.float32, copy=False),
        'raw_difference': raw_difference[valid].astype(np.float32, copy=False),
        'support_difference': reference_difference[valid].astype(
            np.float32, copy=False
        ),
        'pair_ids': pair_ids[valid],
        'valid_count': int(valid.sum()),
        'total_count': int(valid.size),
    }


def binned_pair_variance(
    predictor,
    difference,
    pair_ids,
    selection=None,
    n_bins=N_BINS,
    min_samples=MIN_SAMPLES_PER_BIN,
    max_samples=MAX_SAMPLES,
    fit_quantiles=FIT_QUANTILES,
):
    predictor = np.asarray(predictor).ravel()
    difference = np.asarray(difference).ravel()
    pair_ids = np.asarray(pair_ids).ravel()
    valid = np.isfinite(predictor) & np.isfinite(difference)
    if selection is not None:
        valid &= np.asarray(selection, dtype=bool).ravel()
    if MIN_MEAN_LEVEL is not None:
        valid &= predictor >= MIN_MEAN_LEVEL
    if MAX_MEAN_LEVEL is not None:
        valid &= predictor <= MAX_MEAN_LEVEL

    indices = np.flatnonzero(valid)
    if max_samples is not None and indices.size > max_samples:
        rng = np.random.default_rng(RANDOM_SEED)
        indices = rng.choice(indices, size=int(max_samples), replace=False)
    x = predictor[indices].astype(np.float64, copy=False)
    d = difference[indices].astype(np.float64, copy=False)
    ids = pair_ids[indices]
    if x.size < max(1_000, 3 * min_samples):
        raise ValueError(f'Too few samples remain for a fit: {x.size:,}.')

    lower, upper = np.quantile(x, fit_quantiles)
    central = (x >= lower) & (x <= upper)
    x = x[central]
    d = d[central]
    ids = ids[central]
    edges = np.unique(np.quantile(x, np.linspace(0.0, 1.0, n_bins + 1)))
    if edges.size < 5:
        raise ValueError('The selected predictor has too few distinct levels.')
    bin_ids = np.clip(
        np.searchsorted(edges, x, side='right') - 1, 0, edges.size - 2
    )

    means, variances, errors, counts = [], [], [], []
    for bin_index in range(edges.size - 1):
        in_bin = bin_ids == bin_index
        if int(in_bin.sum()) < min_samples:
            continue
        bin_x = x[in_bin]
        bin_d = d[in_bin]
        bin_pairs = ids[in_bin]
        centered_parts = []
        used_x_parts = []
        degrees_of_freedom = 0
        for pair_id in np.unique(bin_pairs):
            in_pair = bin_pairs == pair_id
            pair_count_in_bin = int(in_pair.sum())
            if pair_count_in_bin < 2:
                continue
            pair_d = bin_d[in_pair]
            centered_parts.append(pair_d - np.mean(pair_d))
            used_x_parts.append(bin_x[in_pair])
            degrees_of_freedom += pair_count_in_bin - 1
        if degrees_of_freedom < 2:
            continue
        centered = np.concatenate(centered_parts)
        used_x = np.concatenate(used_x_parts)
        squared = 0.5 * centered ** 2
        correction = centered.size / degrees_of_freedom
        variance = float(np.mean(squared) * correction)
        error = float(
            np.std(squared, ddof=1) / np.sqrt(squared.size) * correction
        )
        if np.isfinite(variance) and variance > 0:
            means.append(float(np.mean(used_x)))
            variances.append(variance)
            errors.append(error)
            counts.append(centered.size)

    if len(means) < 5:
        raise ValueError('Fewer than five populated variance bins remain.')
    return {
        'mean': np.asarray(means),
        'variance': np.asarray(variances),
        'error': np.asarray(errors),
        'count': np.asarray(counts),
        'fit_limits': (float(lower), float(upper)),
        'sample_count': int(x.size),
    }


def fit_variance_curve(curve):
    models = fit_line_and_quadratic(curve['mean'], curve['variance'])
    if not np.isfinite(models['slope']) or models['slope'] <= 0:
        raise ValueError(
            f"Variance slope must be positive, but it was {models['slope']:.6g}."
        )
    models['photons_per_raw_unit'] = 1.0 / models['slope']
    models['effective_raw_pedestal'] = -models['intercept'] / models['slope']
    return models

In [ ]:
print('Sampling the same spatial coordinates through time...')
raw_trace, support_trace, sampled_coordinates, sampled_flat_indices = (
    sample_matched_traces(raw, support_reference, scale_frame_indices)
)
print(
    f'Sampled trace arrays: {raw_trace.shape[0]} frames x '
    f'{raw_trace.shape[1]:,} spatial positions'
)
pair_raw_trace = raw_trace[pair_trace_indices]
pair_support_trace = (
    support_trace[pair_trace_indices]
    if support_trace is not None
    else None
)

if support_trace is not None:
    intensity_scale = fit_intensity_scale(raw_trace, support_trace)
    scale_gain = intensity_scale['gain']
    scale_offset = intensity_scale['offset']
else:
    intensity_scale = None
    scale_gain = 1.0
    scale_offset = 0.0
    print(
        'WARNING: no SUPPORT reference was supplied. The leave-two-out temporal '
        'mean will be used as the cleaner intensity coordinate.'
    )

pair_data = make_nonoverlap_pair_vectors(
    pair_raw_trace, pair_support_trace, scale_offset, scale_gain
)
legacy_curve = binned_pair_variance(
    pair_data['raw_pair_mean'],
    pair_data['raw_difference'],
    pair_data['pair_ids'],
)
legacy_fit = fit_variance_curve(legacy_curve)

primary_curve = binned_pair_variance(
    pair_data['support_raw_equivalent'],
    pair_data['raw_difference'],
    pair_data['pair_ids'],
    fit_quantiles=PRIMARY_FIT_QUANTILES,
)
primary_fit = fit_variance_curve(primary_curve)

leave_two_out_curve = binned_pair_variance(
    pair_data['leave_two_out'],
    pair_data['raw_difference'],
    pair_data['pair_ids'],
    fit_quantiles=PRIMARY_FIT_QUANTILES,
)
leave_two_out_fit = fit_variance_curve(leave_two_out_curve)

quiet_fit = None
compensated_fit = None
quiet_threshold = None
difference_correlation = np.nan
if support_trace is not None:
    support_change = np.abs(pair_data['support_difference'])
    quiet_threshold = float(
        np.quantile(
            support_change[np.isfinite(support_change)],
            QUIET_SENSITIVITY_QUANTILE,
        )
    )
    quiet_selection = support_change <= quiet_threshold
    quiet_curve = binned_pair_variance(
        pair_data['support_raw_equivalent'],
        pair_data['raw_difference'],
        pair_data['pair_ids'],
        selection=quiet_selection,
        fit_quantiles=PRIMARY_FIT_QUANTILES,
    )
    quiet_fit = fit_variance_curve(quiet_curve)

    compensated_difference = (
        pair_data['raw_difference']
        - pair_data['support_difference'] / scale_gain
    )
    compensated_curve = binned_pair_variance(
        pair_data['support_raw_equivalent'],
        compensated_difference,
        pair_data['pair_ids'],
        fit_quantiles=PRIMARY_FIT_QUANTILES,
    )
    compensated_fit = fit_variance_curve(compensated_curve)
    difference_correlation = correlation(
        pair_data['raw_difference'], pair_data['support_difference']
    )

primary_photons_per_raw_unit = primary_fit['photons_per_raw_unit']
primary_photons_per_support_unit = (
    primary_photons_per_raw_unit / scale_gain
)
effective_support_pedestal = (
    scale_offset
    + scale_gain * primary_fit['effective_raw_pedestal']
)

print()
print('RAW-to-SUPPORT intensity scale (from temporal means at fixed voxels)')
if intensity_scale is None:
    print('  SUPPORT not available; gain=1 and offset=0 are placeholders.')
else:
    print(
        f'  SUPPORT = {scale_offset:.6g} + {scale_gain:.8g} * raw'
    )
    print(f"  linear R^2                 = {intensity_scale['r_squared']:.7f}")
    print(
        '  scale delta AICc (line-quadratic) = '
        f"{intensity_scale['models']['delta_aicc']:+.3f}"
    )

print()
print('MODEL COMPARISON (variance is always measured from raw A-B)')
print(
    f"  legacy noisy-pair axis: p_raw={legacy_fit['photons_per_raw_unit']:.9g}, "
    f"R^2={legacy_fit['linear_r2']:.6f}, "
    f"q={legacy_fit['q2']:+.6g}, "
    f"delta AICc={legacy_fit['delta_aicc']:+.3f}"
)
print(
    f"  SUPPORT intensity axis:   p_raw={primary_photons_per_raw_unit:.9g}, "
    f"R^2={primary_fit['linear_r2']:.6f}, "
    f"q={primary_fit['q2']:+.6g}, "
    f"delta AICc={primary_fit['delta_aicc']:+.3f}"
)
print(
    f"  leave-two-out axis:       p_raw={leave_two_out_fit['photons_per_raw_unit']:.9g}, "
    f"R^2={leave_two_out_fit['linear_r2']:.6f}, "
    f"q={leave_two_out_fit['q2']:+.6g}, "
    f"delta AICc={leave_two_out_fit['delta_aicc']:+.3f}"
)

print()
print('RECOMMENDED SUPPORT-UNIT CONVERSION')
print(f'  photons_per_raw_unit      = {primary_photons_per_raw_unit:.9g}')
print(f'  raw-to-SUPPORT gain       = {scale_gain:.9g}')
print(f'  photons_per_support_unit  = {primary_photons_per_support_unit:.9g}')
print(
    f"  effective raw pedestal    = "
    f"{primary_fit['effective_raw_pedestal']:.6g}"
)
print(f'  effective SUPPORT pedestal= {effective_support_pedestal:.6g}')

print()
print('SENSITIVITY CHECKS (not used as the primary estimate)')
if quiet_fit is not None:
    print(
        f'  quietest {QUIET_SENSITIVITY_QUANTILE:.0%} by SUPPORT change: '
        f"p_support={quiet_fit['photons_per_raw_unit'] / scale_gain:.9g}"
    )
    print(
        '  raw-minus-SUPPORT-change difference: '
        f"p_support={compensated_fit['photons_per_raw_unit'] / scale_gain:.9g}"
    )
    print(
        f'  corr(raw frame difference, SUPPORT frame difference) = '
        f'{difference_correlation:+.6f}'
    )
    if difference_correlation < 0:
        print(
            '  Negative difference correlation means subtraction increases the '
            'measured variance here, so the compensated value is not promoted.'
        )

model_warnings = []
if primary_fit['linear_r2'] < MODEL_R2_WARNING_THRESHOLD:
    model_warnings.append(
        f"primary linear R^2 is below {MODEL_R2_WARNING_THRESHOLD:g}"
    )
if primary_fit['delta_aicc'] > QUADRATIC_AICC_WARNING:
    model_warnings.append(
        'the quadratic model is strongly preferred, so one constant scale is inadequate'
    )
if intensity_scale is not None and intensity_scale['r_squared'] < 0.99:
    model_warnings.append('the raw-to-SUPPORT intensity conversion is not very linear')
if (
    intensity_scale is not None
    and intensity_scale['models']['delta_aicc'] > QUADRATIC_AICC_WARNING
):
    model_warnings.append('the raw-to-SUPPORT scale is meaningfully nonlinear')
if quiet_fit is not None:
    quiet_photons_per_support_unit = (
        quiet_fit['photons_per_raw_unit'] / scale_gain
    )
    quiet_relative_difference = abs(
        quiet_photons_per_support_unit
        / primary_photons_per_support_unit
        - 1.0
    )
    if quiet_relative_difference > MAXIMUM_QUIET_RELATIVE_DIFFERENCE:
        model_warnings.append(
            'quiet-pair photon estimate differs from the primary by '
            f'{quiet_relative_difference:.1%}; motion or biological '
            'dynamics may inflate frame-pair variance'
        )

print()
if model_warnings:
    print('PROVISIONAL RESULT -- review these warnings:')
    for warning in model_warnings:
        print(f'  - {warning}')
else:
    print('The primary linear model passes the configured diagnostics.')
print(f'photons_per_unit={primary_photons_per_support_unit:.9g}')

if known_photons_per_unit is not None:
    relative_error = (
        primary_photons_per_support_unit / known_photons_per_unit - 1.0
    )
    print(f'known synthetic value={known_photons_per_unit:.9g}')
    print(f'synthetic relative error={relative_error:+.2%}')

print()
print('COMPARISON WITH EARLIER VALUES')
print(
    f'  prior raw pair-fit value {PREVIOUS_RAW_PAIR_ESTIMATE:.9g} is '
    f'{PREVIOUS_RAW_PAIR_ESTIMATE / primary_photons_per_raw_unit:.3f}x '
    'the new raw-unit estimate'
)
print(
    f'  prior scale-adjusted value {PREVIOUS_SCALE_ADJUSTED_ESTIMATE:.9g} '
    f'would add '
    f'{primary_photons_per_support_unit / PREVIOUS_SCALE_ADJUSTED_ESTIMATE:.3f}x '
    'the recommended Poisson variance'
)
print(
    f'  current salt.py value {CURRENT_SALT_PHOTONS_PER_UNIT:g} would add '
    f'{primary_photons_per_support_unit / CURRENT_SALT_PHOTONS_PER_UNIT:.3f}x '
    'the recommended Poisson variance'
)

In [ ]:
def draw_variance_panel(axis, predictor, difference, curve, models, title):
    rng = np.random.default_rng(RANDOM_SEED)
    finite = np.flatnonzero(
        np.isfinite(predictor) & np.isfinite(difference)
    )
    if finite.size:
        chosen = rng.choice(
            finite, size=min(25_000, finite.size), replace=False
        )
        axis.scatter(
            np.asarray(predictor)[chosen],
            0.5 * np.asarray(difference)[chosen] ** 2,
            s=3,
            alpha=0.025,
            color='0.35',
            rasterized=True,
            label='sampled pair values',
        )

    axis.errorbar(
        curve['mean'],
        curve['variance'],
        yerr=curve['error'],
        fmt='o',
        capsize=2,
        color='tab:blue',
        label='binned variance',
    )
    line_x = np.linspace(curve['mean'].min(), curve['mean'].max(), 400)
    line_y = models['intercept'] + models['slope'] * line_x
    quadratic_y = (
        models['q0'] + models['q1'] * line_x + models['q2'] * line_x ** 2
    )
    axis.plot(
        line_x,
        line_y,
        color='tab:red',
        linewidth=2.2,
        label=(
            f"linear: p_raw={models['photons_per_raw_unit']:.4g}, "
            f"R2={models['linear_r2']:.4f}"
        ),
    )
    axis.plot(
        line_x,
        quadratic_y,
        '--',
        color='tab:green',
        linewidth=2.0,
        label=(
            f"quadratic: q={models['q2']:+.3g}, "
            f"delta AICc={models['delta_aicc']:+.2f}"
        ),
    )
    if intensity_scale is not None:
        current_raw_ppu = CURRENT_SALT_PHOTONS_PER_UNIT * scale_gain
        axis.plot(
            line_x,
            line_x / current_raw_ppu,
            ':',
            color='tab:purple',
            linewidth=1.8,
            label=f'current salt.py p={CURRENT_SALT_PHOTONS_PER_UNIT:g}',
        )
    axis.set_title(title)
    axis.set_xlabel('Raw-equivalent intensity [image units]')
    axis.set_ylabel('Single-frame raw variance [image units squared]')
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
    upper = max(
        float(np.max(curve['variance'] + curve['error'])),
        float(np.max(line_y)),
        float(np.max(quadratic_y)),
    )
    axis.set_ylim(0, max(upper * 1.15, 1.0))


def draw_residual_panel(axis, curve, models, title):
    linear_residual = curve['variance'] - models['linear_prediction']
    quadratic_residual = curve['variance'] - models['quadratic_prediction']
    axis.errorbar(
        curve['mean'],
        linear_residual,
        yerr=curve['error'],
        fmt='o-',
        capsize=2,
        color='tab:red',
        label='linear residual',
    )
    axis.plot(
        curve['mean'],
        quadratic_residual,
        'o--',
        color='tab:green',
        label='quadratic residual',
    )
    axis.axhline(0.0, color='0.25', linewidth=1)
    axis.set_title(title)
    axis.set_xlabel('Raw-equivalent intensity [image units]')
    axis.set_ylabel('Residual variance')
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)


fig, axes = plt.subplots(2, 3, figsize=(17, 10))
draw_variance_panel(
    axes[0, 0],
    pair_data['raw_pair_mean'],
    pair_data['raw_difference'],
    legacy_curve,
    legacy_fit,
    'A. Legacy: noisy raw pair mean',
)
draw_variance_panel(
    axes[0, 1],
    pair_data['support_raw_equivalent'],
    pair_data['raw_difference'],
    primary_curve,
    primary_fit,
    'B. Recommended: SUPPORT intensity axis',
)
draw_residual_panel(
    axes[1, 0], legacy_curve, legacy_fit, 'D. Legacy-model residuals'
)
draw_residual_panel(
    axes[1, 1], primary_curve, primary_fit, 'E. Recommended-model residuals'
)

scale_axis = axes[0, 2]
scale_residual_axis = axes[1, 2]
if intensity_scale is not None:
    rng = np.random.default_rng(RANDOM_SEED)
    point_count = min(12_000, intensity_scale['raw_means'].size)
    selected = rng.choice(
        intensity_scale['raw_means'].size, size=point_count, replace=False
    )
    scale_axis.scatter(
        intensity_scale['raw_means'][selected],
        intensity_scale['reference_means'][selected],
        s=3,
        alpha=0.04,
        color='0.35',
        rasterized=True,
        label='temporal means at sampled voxels',
    )
    scale_axis.plot(
        intensity_scale['binned_raw'],
        intensity_scale['binned_reference'],
        'o',
        color='tab:blue',
        label='equal-count bins',
    )
    line_x = np.linspace(
        intensity_scale['binned_raw'].min(),
        intensity_scale['binned_raw'].max(),
        400,
    )
    scale_axis.plot(
        line_x,
        scale_offset + scale_gain * line_x,
        color='tab:red',
        linewidth=2,
        label=f'S={scale_offset:.3g}+{scale_gain:.5g} R',
    )
    scale_axis.set_title('C. Temporal-mean raw-to-SUPPORT scale')
    scale_axis.set_xlabel('Mean raw intensity')
    scale_axis.set_ylabel('Mean SUPPORT intensity')
    scale_axis.grid(alpha=0.25)
    scale_axis.legend(fontsize=8)

    scale_residual_axis.plot(
        intensity_scale['binned_raw'],
        intensity_scale['binned_reference']
        - intensity_scale['models']['linear_prediction'],
        'o-',
        color='tab:red',
        label='linear residual',
    )
    scale_residual_axis.plot(
        intensity_scale['binned_raw'],
        intensity_scale['binned_reference']
        - intensity_scale['models']['quadratic_prediction'],
        'o--',
        color='tab:green',
        label='quadratic residual',
    )
    scale_residual_axis.axhline(0.0, color='0.25', linewidth=1)
    scale_residual_axis.set_title('F. Intensity-scale residuals')
    scale_residual_axis.set_xlabel('Mean raw intensity')
    scale_residual_axis.set_ylabel('SUPPORT residual')
    scale_residual_axis.grid(alpha=0.25)
    scale_residual_axis.legend(fontsize=8)
else:
    for axis in (scale_axis, scale_residual_axis):
        axis.text(
            0.5, 0.5, 'No SUPPORT reference', ha='center', va='center'
        )
        axis.set_axis_off()

fig.suptitle(
    'Same raw noise measurement, different estimates of latent intensity',
    fontsize=14,
)
fig.tight_layout()
plt.show()

## How to read the first dashboard

Panels A and D reproduce the old question: “what happens if the noisy raw pair mean is treated as the true intensity?” The result is useful as a failure diagnostic, not as the final conversion.

Panels B and E ask the calibration question directly: “at each denoised SUPPORT intensity, how much raw frame-to-frame variance is present?” A nearly straight curve, structureless residuals, and no meaningful AICc advantage for the quadratic model support one constant <code>photons_per_unit</code>.

Panels C and F establish the unit conversion. The fit is based on temporal means at the same voxels—not instantaneous noisy samples—so noise in a single raw frame does not attenuate the gain as strongly.

The gray clouds are intentionally faint because individual squared differences are very skewed. The blue binned points, model comparison, residuals, and stability plots below carry the inference.

In [ ]:
def fit_selected_pairs(pair_selection):
    selection = np.isin(pair_data['pair_ids'], np.asarray(pair_selection))
    curve = binned_pair_variance(
        pair_data['support_raw_equivalent'],
        pair_data['raw_difference'],
        pair_data['pair_ids'],
        selection=selection,
        n_bins=min(24, N_BINS),
        min_samples=max(100, MIN_SAMPLES_PER_BIN // 2),
        fit_quantiles=PRIMARY_FIT_QUANTILES,
    )
    models = fit_variance_curve(curve)
    return models['photons_per_raw_unit'] / scale_gain


def time_block_estimates():
    unique_pairs = np.unique(pair_data['pair_ids'])
    if TIME_BLOCKS is None or TIME_BLOCKS < 2:
        return np.asarray([]), np.asarray([])
    centers, values = [], []
    for block in np.array_split(
        unique_pairs, min(TIME_BLOCKS, max(1, unique_pairs.size // 2))
    ):
        if block.size < 2:
            continue
        try:
            value = fit_selected_pairs(block)
        except ValueError:
            continue
        center_frame = float(np.mean(pair_starts[block])) + 0.5
        centers.append(center_frame)
        values.append(value)
    return np.asarray(centers), np.asarray(values)


def lag_estimates():
    rng = np.random.default_rng(RANDOM_SEED + 1)
    coordinate_count = min(LAG_COORDINATE_COUNT, raw_trace.shape[1])
    coordinates = np.sort(
        rng.choice(raw_trace.shape[1], coordinate_count, replace=False)
    )
    lag_values, estimates = [], []
    for lag in DIAGNOSTIC_LAGS:
        if lag >= raw_trace.shape[0] - 2:
            continue
        raw_a = raw_trace[:-lag, coordinates]
        raw_b = raw_trace[lag:, coordinates]
        difference = raw_a - raw_b
        if support_trace is not None:
            reference_level = 0.5 * (
                support_trace[:-lag, coordinates]
                + support_trace[lag:, coordinates]
            )
            predictor = (reference_level - scale_offset) / scale_gain
        else:
            predictor = np.broadcast_to(
                temporal_mean(raw_trace[:, coordinates])[None, :],
                difference.shape,
            )
        valid = (
            np.isfinite(raw_a)
            & np.isfinite(raw_b)
            & np.isfinite(predictor)
            & np.isfinite(difference)
        )
        if SATURATION_VALUE is not None:
            valid &= (raw_a < SATURATION_VALUE) & (raw_b < SATURATION_VALUE)
        if FLOOR_VALUE is not None:
            valid &= (raw_a > FLOOR_VALUE) & (raw_b > FLOOR_VALUE)
        pair_ids = np.broadcast_to(
            np.arange(difference.shape[0], dtype=np.int32)[:, None],
            difference.shape,
        )
        try:
            curve = binned_pair_variance(
                predictor[valid],
                difference[valid],
                pair_ids[valid],
                n_bins=min(24, N_BINS),
                min_samples=max(100, MIN_SAMPLES_PER_BIN // 2),
                fit_quantiles=PRIMARY_FIT_QUANTILES,
            )
            models = fit_variance_curve(curve)
        except ValueError:
            continue
        lag_values.append(lag)
        estimates.append(models['photons_per_raw_unit'] / scale_gain)
    return np.asarray(lag_values), np.asarray(estimates)


def local_group_estimate(coordinate_mask):
    coordinate_mask = np.asarray(coordinate_mask, dtype=bool)
    if int(coordinate_mask.sum()) < 100:
        return np.nan
    if support_trace is not None:
        local_scale = fit_intensity_scale(
            raw_trace,
            support_trace,
            coordinate_mask=coordinate_mask,
            n_bins=min(15, N_BINS),
            min_samples=10,
        )
        local_gain = local_scale['gain']
        local_offset = local_scale['offset']
    else:
        local_gain = 1.0
        local_offset = 0.0
    local_pairs = make_nonoverlap_pair_vectors(
        pair_raw_trace[:, coordinate_mask],
        (
            pair_support_trace[:, coordinate_mask]
            if pair_support_trace is not None
            else None
        ),
        local_offset,
        local_gain,
    )
    try:
        curve = binned_pair_variance(
            local_pairs['support_raw_equivalent'],
            local_pairs['raw_difference'],
            local_pairs['pair_ids'],
            n_bins=min(20, N_BINS),
            min_samples=100,
            fit_quantiles=PRIMARY_FIT_QUANTILES,
        )
        models = fit_variance_curve(curve)
    except ValueError:
        return np.nan
    return models['photons_per_raw_unit'] / local_gain


def grouped_spatial_estimates():
    if len(sampled_coordinates) == 3:
        z_coordinates = np.asarray(sampled_coordinates[0])
    else:
        z_coordinates = np.zeros(raw_trace.shape[1], dtype=np.int32)
    y_coordinates = np.asarray(sampled_coordinates[-2])
    x_coordinates = np.asarray(sampled_coordinates[-1])

    z_labels, z_values = [], []
    for z_index in np.unique(z_coordinates):
        value = local_group_estimate(z_coordinates == z_index)
        if np.isfinite(value):
            z_labels.append(int(z_index))
            z_values.append(value)

    tile_rows, tile_columns = TILE_GRID
    tile_values = np.full((tile_rows, tile_columns), np.nan)
    y_edges = np.linspace(0, raw.shape[-2], tile_rows + 1)
    x_edges = np.linspace(0, raw.shape[-1], tile_columns + 1)
    for row in range(tile_rows):
        for column in range(tile_columns):
            selected = (
                (y_coordinates >= y_edges[row])
                & (y_coordinates < y_edges[row + 1])
                & (x_coordinates >= x_edges[column])
                & (x_coordinates < x_edges[column + 1])
            )
            tile_values[row, column] = local_group_estimate(selected)
    return np.asarray(z_labels), np.asarray(z_values), tile_values


def empty_moments():
    return {
        'n': 0,
        'sx': 0.0,
        'sy': 0.0,
        'sxx': 0.0,
        'syy': 0.0,
        'sxy': 0.0,
    }


def update_moments(moment, x, y):
    x = np.asarray(x, dtype=np.float64).ravel()
    y = np.asarray(y, dtype=np.float64).ravel()
    valid = np.isfinite(x) & np.isfinite(y)
    if not np.any(valid):
        return
    x = x[valid]
    y = y[valid]
    moment['n'] += x.size
    moment['sx'] += float(np.sum(x))
    moment['sy'] += float(np.sum(y))
    moment['sxx'] += float(np.dot(x, x))
    moment['syy'] += float(np.dot(y, y))
    moment['sxy'] += float(np.dot(x, y))


def moments_correlation(moment):
    n = moment['n']
    if n < 3:
        return np.nan
    covariance = moment['sxy'] - moment['sx'] * moment['sy'] / n
    variance_x = moment['sxx'] - moment['sx'] ** 2 / n
    variance_y = moment['syy'] - moment['sy'] ** 2 / n
    denominator = np.sqrt(max(variance_x * variance_y, 0.0))
    return float(covariance / denominator) if denominator > 0 else np.nan


def center_each_plane(array):
    array = np.asarray(array, dtype=np.float32)
    if array.ndim == 2:
        return array - np.nanmean(array)
    axes = tuple(range(1, array.ndim))
    plane_means = np.nanmean(array, axis=axes, keepdims=True)
    return array - plane_means


def spatial_difference_correlations():
    raw_x = [empty_moments() for _ in range(SPATIAL_CORRELATION_LAGS)]
    raw_y = [empty_moments() for _ in range(SPATIAL_CORRELATION_LAGS)]
    compensated_x = [
        empty_moments() for _ in range(SPATIAL_CORRELATION_LAGS)
    ]
    compensated_y = [
        empty_moments() for _ in range(SPATIAL_CORRELATION_LAGS)
    ]

    chosen_pairs = np.unique(
        np.linspace(
            0, pair_count - 1, min(SPATIAL_PAIR_COUNT, pair_count), dtype=int
        )
    )
    for sequence_index, pair_index in enumerate(chosen_pairs):
        first_index = int(pair_starts[int(pair_index)])
        raw_difference = (
            np.asarray(raw[first_index], dtype=np.float32)
            - np.asarray(raw[first_index + 1], dtype=np.float32)
        )
        raw_difference = center_each_plane(raw_difference)
        if support_reference is not None:
            support_difference = (
                np.asarray(support_reference[first_index], dtype=np.float32)
                - np.asarray(support_reference[first_index + 1], dtype=np.float32)
            )
            compensated = center_each_plane(
                raw_difference - support_difference / scale_gain
            )
        else:
            compensated = raw_difference

        for lag in range(1, SPATIAL_CORRELATION_LAGS + 1):
            update_moments(
                raw_x[lag - 1],
                raw_difference[..., :-lag],
                raw_difference[..., lag:],
            )
            update_moments(
                raw_y[lag - 1],
                raw_difference[..., :-lag, :],
                raw_difference[..., lag:, :],
            )
            update_moments(
                compensated_x[lag - 1],
                compensated[..., :-lag],
                compensated[..., lag:],
            )
            update_moments(
                compensated_y[lag - 1],
                compensated[..., :-lag, :],
                compensated[..., lag:, :],
            )
        print(
            f'  spatial diagnostic pair {sequence_index + 1}/{chosen_pairs.size}',
            end='\r' if sequence_index + 1 < chosen_pairs.size else '\n',
        )

    return {
        'raw_x': np.asarray([moments_correlation(x) for x in raw_x]),
        'raw_y': np.asarray([moments_correlation(x) for x in raw_y]),
        'compensated_x': np.asarray(
            [moments_correlation(x) for x in compensated_x]
        ),
        'compensated_y': np.asarray(
            [moments_correlation(x) for x in compensated_y]
        ),
    }


def darkest_border_proxy():
    if support_trace is None:
        return np.asarray([]), np.asarray([])
    y_coordinates = np.asarray(sampled_coordinates[-2])
    x_coordinates = np.asarray(sampled_coordinates[-1])
    border_y = max(1, int(np.ceil(raw.shape[-2] * DARK_BORDER_FRACTION)))
    border_x = max(1, int(np.ceil(raw.shape[-1] * DARK_BORDER_FRACTION)))
    border = (
        (y_coordinates < border_y)
        | (y_coordinates >= raw.shape[-2] - border_y)
        | (x_coordinates < border_x)
        | (x_coordinates >= raw.shape[-1] - border_x)
    )
    if len(sampled_coordinates) == 3:
        z_coordinates = np.asarray(sampled_coordinates[0])
    else:
        z_coordinates = np.zeros(raw_trace.shape[1], dtype=np.int32)
    reference_means = temporal_mean(support_trace)
    labels, values = [], []
    for z_index in np.unique(z_coordinates):
        selected = border & (z_coordinates == z_index)
        candidates = reference_means[selected]
        candidates = candidates[np.isfinite(candidates)]
        if candidates.size < 10:
            continue
        threshold = np.quantile(candidates, DARK_BORDER_QUANTILE)
        darkest = candidates[candidates <= threshold]
        labels.append(int(z_index))
        values.append(float(np.median(darkest)))
    return np.asarray(labels), np.asarray(values)

In [ ]:
print('Running stability and correlation diagnostics...')
block_centers, block_ppu = time_block_estimates()
lag_values, lag_ppu = lag_estimates()
z_labels, z_ppu, tile_ppu = grouped_spatial_estimates()

raw_adjacent_correlation = correlation(raw_trace[:-1], raw_trace[1:])
if support_trace is not None:
    support_adjacent_correlation = correlation(
        support_trace[:-1], support_trace[1:]
    )
    raw_support_same_frame_correlation = correlation(
        raw_trace, support_trace
    )
else:
    support_adjacent_correlation = np.nan
    raw_support_same_frame_correlation = np.nan

spatial_correlations = spatial_difference_correlations()
background_z, background_support = darkest_border_proxy()

print()
print('TIME AND SPACE STABILITY')
if block_ppu.size:
    print(
        f'  time blocks: {block_ppu.min():.9g} to {block_ppu.max():.9g} '
        f'(median {np.median(block_ppu):.9g})'
    )
if lag_ppu.size:
    print('  temporal lags:')
    for lag, value in zip(lag_values, lag_ppu):
        print(f'    lag {lag:>2}: p_support={value:.9g}')
if z_ppu.size:
    print(
        f'  Z slices: {np.nanmin(z_ppu):.9g} to {np.nanmax(z_ppu):.9g}'
    )
if np.any(np.isfinite(tile_ppu)):
    print(
        f'  XY tiles: {np.nanmin(tile_ppu):.9g} to '
        f'{np.nanmax(tile_ppu):.9g}'
    )

print()
print('CORRELATION DIAGNOSTICS')
print(f'  adjacent raw-frame correlation     = {raw_adjacent_correlation:+.6f}')
print(
    f'  adjacent SUPPORT-frame correlation = '
    f'{support_adjacent_correlation:+.6f}'
)
print(
    f'  same-frame raw/SUPPORT correlation = '
    f'{raw_support_same_frame_correlation:+.6f}'
)
print(
    f'  raw/SUPPORT first-difference corr   = '
    f'{difference_correlation:+.6f}'
)
print(
    '  raw difference spatial lag 1: '
    f"X={spatial_correlations['raw_x'][0]:+.6f}, "
    f"Y={spatial_correlations['raw_y'][0]:+.6f}"
)
print(
    '  compensated sensitivity lag 1: '
    f"X={spatial_correlations['compensated_x'][0]:+.6f}, "
    f"Y={spatial_correlations['compensated_y'][0]:+.6f}"
)
if background_support.size:
    print(
        f'  darkest-border SUPPORT proxy: {background_support.min():.6g} to '
        f'{background_support.max():.6g}; '
        f'median {np.median(background_support):.6g}'
    )
    print(
        f'  fitted effective SUPPORT pedestal: '
        f'{effective_support_pedestal:.6g}'
    )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axis = axes[0, 0]
if block_ppu.size:
    axis.plot(block_centers, block_ppu, 'o-', color='tab:blue')
    axis.axhline(
        primary_photons_per_support_unit,
        color='tab:red',
        linestyle='--',
        label='full fit',
    )
    axis.legend()
else:
    axis.text(0.5, 0.5, 'Not enough data for blocks', ha='center')
axis.set_title('A. Time-block stability')
axis.set_xlabel('Center raw-frame number')
axis.set_ylabel('photons_per_SUPPORT_unit')
axis.grid(alpha=0.25)

axis = axes[0, 1]
if lag_ppu.size:
    axis.plot(lag_values, lag_ppu, 'o-', color='tab:blue')
    axis.axhline(
        primary_photons_per_support_unit,
        color='tab:red',
        linestyle='--',
        label='non-overlapping adjacent-pair fit',
    )
    axis.legend()
else:
    axis.text(0.5, 0.5, 'No lag estimates', ha='center')
axis.set_title('B. Temporal-lag sensitivity')
axis.set_xlabel('Frame lag')
axis.set_ylabel('photons_per_SUPPORT_unit')
axis.grid(alpha=0.25)

axis = axes[1, 0]
if z_ppu.size:
    axis.plot(z_labels, z_ppu, 'o-', color='tab:blue')
    axis.axhline(
        primary_photons_per_support_unit,
        color='tab:red',
        linestyle='--',
        label='full fit',
    )
    axis.legend()
else:
    axis.text(0.5, 0.5, 'No Z estimates', ha='center')
axis.set_title('C. Per-Z estimate with local scale conversion')
axis.set_xlabel('Z index after ROI/stride')
axis.set_ylabel('photons_per_SUPPORT_unit')
axis.grid(alpha=0.25)

axis = axes[1, 1]
shown = axis.imshow(tile_ppu, cmap='viridis', aspect='auto')
for row in range(tile_ppu.shape[0]):
    for column in range(tile_ppu.shape[1]):
        value = tile_ppu[row, column]
        if np.isfinite(value):
            axis.text(
                column,
                row,
                f'{value:.3g}',
                ha='center',
                va='center',
                color='white' if value < np.nanmedian(tile_ppu) else 'black',
                fontsize=8,
            )
axis.set_title('D. XY-tile estimates with local scale conversion')
axis.set_xlabel('X tile')
axis.set_ylabel('Y tile')
fig.colorbar(shown, ax=axis, label='photons_per_SUPPORT_unit')

fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
lags = np.arange(1, SPATIAL_CORRELATION_LAGS + 1)
axes[0].plot(
    lags, spatial_correlations['raw_x'], 'o-', label='raw difference, X'
)
axes[0].plot(
    lags, spatial_correlations['raw_y'], 'o-', label='raw difference, Y'
)
axes[0].plot(
    lags,
    spatial_correlations['compensated_x'],
    'o--',
    label='SUPPORT-change sensitivity, X',
)
axes[0].plot(
    lags,
    spatial_correlations['compensated_y'],
    'o--',
    label='SUPPORT-change sensitivity, Y',
)
axes[0].axhline(0.0, color='0.25', linewidth=1)
axes[0].set_title('Spatial correlation of frame differences')
axes[0].set_xlabel('Pixel lag')
axes[0].set_ylabel('Correlation')
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)

if background_support.size:
    axes[1].plot(
        background_z,
        background_support,
        'o-',
        color='tab:blue',
        label='darkest sampled border proxy',
    )
    axes[1].axhline(
        effective_support_pedestal,
        color='tab:red',
        linestyle='--',
        label='fit effective pedestal',
    )
    axes[1].legend(fontsize=8)
else:
    axes[1].text(
        0.5, 0.5, 'No SUPPORT border proxy', ha='center', va='center'
    )
axes[1].set_title('Background proxy versus fitted pedestal')
axes[1].set_xlabel('Z index after ROI/stride')
axes[1].set_ylabel('SUPPORT intensity')
axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()

## Decision guide and limitations

Use the value printed as <code>photons_per_support_unit</code> for the denoised SUPPORT image only when all of the following are reasonably true:

- The recommended SUPPORT-axis curve is visually close to a line, its residuals have little structure, and the quadratic model does not win by $\Delta\mathrm{AICc}>6$.
- Time blocks, temporal lags, Z slices, and XY tiles are acceptably stable for the intended simulation.
- The raw-to-SUPPORT temporal-mean conversion is close to affine.
- Clipping and saturation have been excluded using the actual detector limits.

Why the sensitivity fits are not automatically better:

- Raw $A-B$ can be too variable if biological change or motion remains.
- $(A-B)-(S_A-S_B)/g$ can be too quiet if the denoiser leaks raw noise into SUPPORT. A negative raw/SUPPORT difference correlation makes that subtraction increase variance instead. The notebook therefore keeps raw $A-B$ as the primary measurement and shows quiet-change and compensated results as brackets.
- SUPPORT is derived from the same acquisition, so this is an empirical effective calibration—not an independent detector calibration.

The spatial-correlation plot matters to <code>salt.py</code>: independent Poisson draws can match the marginal variance but cannot reproduce correlated horizontal or vertical detector noise. A large lag-1 value means the simulated texture will still differ even with the right scalar count scale.

The darkest-border curve is not a dark measurement. If it differs from the fitted effective pedestal, do not “fix” the intercept by subtracting the border value. Acquire laser-off/dark frames to separate the digital offset and read variance. For a formal detector calibration, use uniform flat-field pairs at several illumination levels plus dark frames and follow the [EMVA 1288 photon-transfer procedure](https://www.emva.org/wp-content/uploads/EMVA1288Linear_4.0Release.pdf).

### Relation to Stefanov (IEEE TED, 2014)

Stefanov's signal-dependent charge-sharing model is relevant to the *shape test*: it adds a negative quadratic contribution to variance versus signal. If the fitted $q$ is convincingly negative and the quadratic AICc advantage is strong, a constant <code>photons_per_unit</code> is incomplete. The earlier plot's pronounced **positive** curvature has the opposite sign and largely disappears when SUPPORT supplies the intensity axis, so that particular curvature is not evidence for Stefanov charge sharing.

Finally, scaling every stored intensity by $k$ divides <code>photons_per_unit</code> by $k$. Estimate the value separately for every detector gain, channel, preprocessing scale, and SUPPORT model. The returned number is intended for the lower-power SUPPORT-image simulation path—not for repeatedly lowering and re-noising the raw movie.

In [ ]:
print('Copy-ready setting for the SUPPORT-image simulation:')
print(f'photons_per_unit = {primary_photons_per_support_unit:.9g}')
if model_warnings:
    print('Status: provisional; inspect the warnings and dashboards above.')
else:
    print('Status: supported by the configured linearity checks.')